# Как из временного ряда получить осмысленные признаки

В энергетических задачах таблица наблюдений редко бывает самодостаточной. Чаще всего она содержит лишь исходные измерения, а содержательные признаки приходится конструировать так, чтобы они отражали логику процесса: суточный ход, инерцию нагрузки, влияние предшествующих значений.

<div style="border-left: 4px solid #1f4b99; background: #eef5ff; padding: 12px 14px; margin: 12px 0;">
<strong>Учебная задача.</strong><br/>
На основе реального временного ряда потребления мощности необходимо показать, как из исходных измерений формируются признаки, пригодные для анализа и прогноза: календарные характеристики, лаги и скользящие агрегаты.
</div>

## Исходные данные

В блокноте используется локальная учебная выборка, подготовленная из датасета <em>Individual Household Electric Power Consumption</em>. Для удобства работы она уже агрегирована до часового шага и содержит признаки, отражающие структуру временного ряда.

## Что должно получиться

После выполнения блокнота должно быть понятно:

- как выглядит реальный профиль нагрузки во времени;
- какие признаки можно извлечь из временной структуры;
- почему лаги и скользящие характеристики полезны для прогноза;
- как перейти от таблицы измерений к признаковому пространству.

## Ход анализа

### 1. Открытие локального ряда наблюдений
Загрузка выполняется из `data/sample/time_series/` через относительный путь.

### 2. Первичное описание временного ряда
Проверяются границы периода, шаг и базовые статистики нагрузки.

### 3. Построение производных признаков
Изучаются календарные, лаговые и агрегированные характеристики.

### 4. Интерпретация
Оценивается, какие признаки могут быть полезны для последующего прогноза.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from src.display_utils import display_callout, display_metric_table
from src.project_paths import sample_data_dir

DATA_PATH = sample_data_dir() / "time_series" / "household_power_hourly_january_2007.csv"
df = pd.read_csv(DATA_PATH, parse_dates=["timestamp"])
df.head()

## Первое наблюдение

В отличие от синтетического примера здесь используются реальные наблюдения, агрегированные в часовом масштабе. Уже по структуре столбцов видно, что часть признаков относится к исходным измерениям, а часть — к специально подготовленным характеристикам временного ряда.

In [ ]:
period_summary = pd.DataFrame(
    {
        "start": [df["timestamp"].min()],
        "end": [df["timestamp"].max()],
        "rows": [len(df)],
        "missing_load": [int(df["Global_active_power"].isna().sum())],
    }
)
period_summary

## Интерпретация периода наблюдений

Период охватывает январь 2007 года в часовом разрешении. Такая выборка достаточно компактна для учебной работы, но при этом сохраняет временную структуру процесса и позволяет демонстрировать построение признаков на реальных данных.

In [ ]:
stats = df[["Global_active_power", "Voltage", "Global_intensity"]].describe().T
stats

## Интерпретация описательных статистик

Описательные статистики помогают оценить диапазон колебаний нагрузки и сопутствующих параметров. Они служат основой для проверки выбросов, выбора масштаба визуализации и интерпретации сезонных и суточных закономерностей.

In [ ]:
hourly_profile = df.groupby("hour", as_index=False)["Global_active_power"].mean()
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(hourly_profile["hour"], hourly_profile["Global_active_power"], marker="o")
ax.set_title("Средний суточный профиль активной мощности")
ax.set_xlabel("Час суток")
ax.set_ylabel("Global_active_power")
ax.grid(True, alpha=0.3)
plt.show()

## Что показывает суточный профиль

Даже простое усреднение по часу суток позволяет увидеть внутреннюю структуру процесса. Для задач прогноза это важно, поскольку временной ряд несет в себе не только последовательность наблюдений, но и выраженный календарный ритм.

In [ ]:
feature_view = df[[
    "timestamp",
    "Global_active_power",
    "hour",
    "day_of_week",
    "load_lag_1",
    "load_roll_3",
]].head(10)
feature_view

## Интерпретация признаков

Временной ряд был преобразован в набор признаков, которые можно подавать на вход модели. Календарные признаки отражают место наблюдения во времени, а лаговые и скользящие характеристики описывают инерцию процесса.

In [ ]:
correlation = df[[
    "Global_active_power",
    "hour",
    "day_of_week",
    "load_lag_1",
    "load_roll_3",
]].corr(numeric_only=True)
correlation

## Зачем нужна корреляционная матрица

Она не заменяет полноценный анализ, но позволяет быстро увидеть, какие признаки ближе всего связаны с текущей нагрузкой. В данном примере особенно показательны лаг и скользящее среднее, что согласуется с инерционной природой временного процесса.

In [ ]:
display_metric_table(
    {
        "mean_load": float(df["Global_active_power"].mean()),
        "std_load": float(df["Global_active_power"].std()),
        "non_null_lag_share": float(df["load_lag_1"].notna().mean()),
        "non_null_roll_share": float(df["load_roll_3"].notna().mean()),
    },
    decimals=3,
)

display_callout(
    "Вывод по признаковому пространству",
    "Реальный временной ряд уже на базовом уровне позволяет построить календарные, лаговые и агрегированные признаки, которые далее могут использоваться в прогнозной модели.",
    tone="success",
)

## Итоговый вывод

Даже относительно короткий фрагмент реального временного ряда позволяет построить осмысленное признаковое пространство. Это подтверждает, что разведочный анализ является не вспомогательной иллюстрацией, а обязательным переходом от данных к модели.

## Вопросы для самостоятельной работы

1. Добавьте признак выходного дня и оцените, как он соотносится с нагрузкой.
2. Почему лаговые признаки нельзя строить без учета направления времени?
3. Какие еще агрегированные характеристики были бы полезны для краткосрочного прогноза?

## Источники

1. [Глава 3 пособия](https://github.com/Nephalem72/power-data-analysis-handbook/blob/main/docs/chapters/03-eda-and-features.md)
2. [Паспорт датасетов](https://github.com/Nephalem72/power-data-analysis-handbook/blob/main/docs/datasets.md)